# 划分数据集（GTA）

In [2]:
import os
from Bio import SeqIO

# 生成替身序列，防止cdhit出毛病

original_fasta = './data/cluster/GTA/sequences.fasta'
index_fasta = './data/cluster/GTA/index_fasta.fasta'
index_dict = './data/cluster/GTA/index_dict.dict'

with open(original_fasta, "r") as f:
    sequences = SeqIO.parse(f, "fasta")
    sequence_dict = {seq_record.id : seq_record.seq for seq_record in sequences}

# 生成替身序列，并记录替身字典
f = open(index_fasta, 'w')
f_dict = open(index_dict, 'w')
key_list = list(sequence_dict.keys())
for i in range(0,len(key_list)):
    f_dict.write("seq{:06d}==={}\n".format(i, key_list[i]))
    f.write(">seq{:06d}\n{}\n".format(i, sequence_dict[key_list[i]]))
f.close()
f_dict.close()

print(f"cdhit -c 0.7 -i index_fasta.fasta -o index_fasta_cdhit.fasta -T 192 -M 20000")

# 在/home/admin123/work/GTmining/data/cluster/GTA/下运行


cdhit -c 0.7 -i index_fasta.fasta -o index_fasta_cdhit.fasta -T 192 -M 20000


In [83]:
# 采样获取训练集：验证集：测试集=7：2：1，单独取样nova数据集（成家族取样）
import os
import pandas as pd
import random
from tqdm import tqdm

cluster_file = './data/cluster/GTA/index_fasta_cdhit.fasta.clstr'
index_dict_file = './data/cluster/GTA/index_dict.dict'
nova_family = ['GT7', 'GT25', 'GT55', 'GT64', 'GT81', 'GT82'] # GTA

# 读取家族信息
df = pd.read_excel('./data/GTA_training_data.xlsx')
family_dict = {}
for i in range(0, df.shape[0]):
    family_dict[df['Family'][i]+'_'+df['GenBank'][i]] = df['Family'][i]

# 活性标签
activate_index = {'UDP-Glc': 0, 'UDP-GlcNAc': 1, 'UDP-GlcA': 2,
                  'UDP-Gal': 3, 'UDP-GalNAc': 4,
                  'UDP-Xyl': 5, 'GDP-Man': 6,
                  'dTDP-Rha': 7, 'Other': 8}
# activate_index = {'UDP-Glc': 0, 'UDP-GlcNAc': 1, 'UDP-GlcA': 2,
#                   'UDP-Gal': 3, 'UDP-GalNAc': 4,
#                   'UDP-Xyl': 5, 'GDP-Man': 6, 'GDP-Fuc': 7,
#                   'dTDP-Rha': 8, 'Other': 9}


# 获取活性标签
df = pd.read_excel('./data/GTA_training_data.xlsx')
activate_dict = {}
for i in range(0,df.shape[0]):
    if '[-]' in df['NDP-Sugar活性'][i]:
        activate_dict[df['Family'][i]+'_'+df['GenBank'][i]] = 'Bifunction'
    elif '[*]' in df['NDP-Sugar活性'][i]:
        activate_dict[df['Family'][i]+'_'+df['GenBank'][i]] = 'Other'
    else:
        activate_dict[df['Family'][i]+'_'+df['GenBank'][i]] = df['NDP-Sugar活性'][i]

# 读取cluster文件
# 读取index文件
index_dict = {}
with open(index_dict_file, 'r')as f:
    for line in f:
        line = line.strip().split('===')
        index_dict[line[0]] = line[1]
# 转译cluster文件
clusters = {}
with open(cluster_file, 'r') as file:
    current_cluster = None
    for line in file:
        line = line.strip()
        if line.startswith('>Cluster'):
            # 新的cluster开始
            cluster_number = 'Cluster_' + line.split()[1]
            current_cluster = cluster_number
            clusters[current_cluster] = []
        elif line and current_cluster is not None:
            # 这一行包含序列信息
            seq_name = line.split(', >')[1].split('... ')[0]
            clusters[current_cluster].append(index_dict[seq_name])

# 检查cluster中活性数据是否冲突
for _, cluster_list in clusters.items():
    cluster_activate = [activate_dict[x] for x in cluster_list]
    cluster_activate = list(set(cluster_activate))
    # if len(cluster_activate) >= 2:
    #     print('┗|｀O′|┛ 嗷~~，发现有的簇花心了！活性不对。')
    #     print(cluster_activate, cluster_list)

# 检查cluster中家族数据是否冲突
for _, cluster_list in clusters.items():
    cluster_activate = [family_dict[x] for x in cluster_list]
    cluster_activate = list(set(cluster_activate))
    # if len(cluster_activate) >= 2:
    #     print('┗|｀O′|┛ 嗷~~，发现有的簇花心了！家族不对。')
    #     print(cluster_activate, cluster_list)

cluster_dict = {x : [] for x in activate_index.keys()}
# 写入活性cluter字典
for _, cluster_list in clusters.items():
    if activate_dict[cluster_list[0]] == 'Bifunction':
        # 双功能的酶暂时不在考虑之中，因为在现有的体系里面划分到哪一类都不是很合适
        # 之前划分到other类里面可能会导致模型在学习的时候有点分不清楚
        continue
    elif activate_dict[cluster_list[0]] == 'UDP-GalA' or activate_dict[cluster_list[0]] == 'GDP-Fuc':
        # GalA因为数量实在是太少，所以不在考虑的范围之中
        continue
    elif len(list(set([activate_dict[x] for x in cluster_list]))) >= 2:
        # 有的序列有两个结构域，并且是双功能
        continue
    elif len(list(set([family_dict[x] for x in cluster_list]))) >= 2:
        # 有的序列有两个结构域，并且是双功能
        continue
    cluster_dict[activate_dict[cluster_list[0]]].append([x for x in cluster_list])



fold_num = 1
while fold_num <= 10:
    # ==================== 划分训练集、测试集 ====================
    df = pd.DataFrame(columns=['Family', 'GenBank', 'Cluster_GenBank_Random', 'Activate', 'Dataset'])

    for temp_key, temp_value in cluster_dict.items():
        # 写入realtest
        new_value = []
        realtest_value = []
        for cluster in temp_value:
            if cluster[0].split('_')[0] in nova_family:
                realtest_value.append(cluster)
            else:
                new_value.append(cluster)
        temp_value = new_value
        realtest_samples = realtest_value
        # 处理realtest
        for samples in tqdm(realtest_samples, desc=temp_key+'_nova'):
            sample_family = samples[0].split('_')[0]
            for sample in samples:
                df.loc[len(df)] = [sample_family, sample, samples[0], activate_dict[sample], 'nova']


        # 计算各集合的目标范围（总样本数的±5%）
        total_samples = sum(len(c) for c in temp_value)
        train_min = 0.68 * total_samples
        train_max = 0.72 * total_samples
        validation_min = 0.18 * total_samples
        validation_max = 0.22 * total_samples
        test_min = 0.08 * total_samples
        test_max = 0.12 * total_samples

        total_attempts = 0
        while total_attempts < 20:
            # 随机打乱簇的顺序，之后就可以直接遍历而无需采样
            shuffled = random.sample(temp_value, len(temp_value))
            
            # 初始化集合
            train, validation, test = [], [], []
            train_size, validation_size, test_size = 0, 0, 0
            
            for cluster in shuffled:
                cluster_size = len(cluster)
                candidates = []
                
                # 确定可加入的集合
                if test_size + cluster_size <= test_max:
                    candidates.append('test')
                if validation_size + cluster_size <= validation_max:
                    candidates.append('validation')
                if train_size + cluster_size <= train_max:
                    candidates.append('train')
                
                if not candidates:
                    break  # 无法分配，本次尝试失败
                
                # 优先填充未达最小值的集合
                priorities = []
                for candidate in candidates:
                    current = {'train': train_size, 'validation': validation_size, 'test': test_size}[candidate]
                    min_req = {'train': train_min, 'validation': validation_min, 'test': test_min}[candidate]
                    priority = min_req - current if current < min_req else 100000
                    priorities.append((priority, candidate))
                
                # 选择最需要填充的集合（谁差的最少优先选谁）
                priorities.sort(reverse=False, key=lambda x: x[0])
                chosen = priorities[0][1]

                # 分配簇
                if chosen == 'train':
                    train.append(cluster)
                    train_size += cluster_size
                elif chosen == 'validation':
                    validation.append(cluster)
                    validation_size += cluster_size
                elif chosen == 'test':
                    test.append(cluster)
                    test_size += cluster_size

            
            # 验证分配结果
            if (train_min <= train_size <= train_max and
                validation_min <= validation_size <= validation_max and
                test_min <= test_size <= test_max and
                (train_size + validation_size + test_size) == total_samples):
                # 展开簇中的样本
                train_samples = train
                validation_samples = validation
                test_samples = test
                break
            
            total_attempts += 1

        if total_attempts >= 20:
            print(f"error in {temp_key}")
            break
        
        for samples in tqdm(train_samples, desc=temp_key + ' train'):
            sample_family = samples[0].split('_')[0]
            for sample in samples:
                if activate_dict[sample] == 'Bifunction':
                    print(sample)
                df.loc[len(df)] = [sample_family, sample, samples[0], activate_dict[sample], 'train']
        for samples in tqdm(validation_samples, desc=temp_key + ' validation'):
            sample_family = samples[0].split('_')[0]
            for sample in samples:
                if activate_dict[sample] == 'Bifunction':
                    print(sample)
                df.loc[len(df)] = [sample_family, sample, samples[0], activate_dict[sample], 'validation']
        for samples in tqdm(test_samples, desc=temp_key + ' test'):
            sample_family = samples[0].split('_')[0]
            for sample in samples:
                if activate_dict[sample] == 'Bifunction':
                    print(sample)
                df.loc[len(df)] = [sample_family, sample, samples[0], activate_dict[sample], 'test']
        
        # break
        
    df = df.drop_duplicates()
    df.reset_index(drop=True, inplace=True)

    split_name = [df['Activate'][i] + '_' + df['Dataset'][i] for i in range(0, df.shape[0])]
    if not len(list(set(split_name))) == 35:
        print('========== ┗|｀O′|┛ 嗷~~，分类中有类缺失！ ==========')
    else:
        print('========== 分类中没有类缺失！ ==========')
        df.to_excel(f'./data/cluster/GTA/dataseat_split_{fold_num}.xlsx', index=False)
        fold_num += 1

GDP-Man test: 100%|██████████| 43/43 [00:00<00:00, 122.10it/s]
dTDP-Rha_nova: 0it [00:00, ?it/s]
Other test: 100%|██████████| 56/56 [00:00<00:00, 144.09it/s]


========== 分类中没有类缺失！ ==========


GDP-Man test: 100%|██████████| 59/59 [00:00<00:00, 150.68it/s]
dTDP-Rha_nova: 0it [00:00, ?it/s]
Other test: 100%|██████████| 38/38 [00:00<00:00, 97.96it/s]


========== 分类中没有类缺失！ ==========


GDP-Man test: 100%|██████████| 29/29 [00:00<00:00, 74.52it/s]
dTDP-Rha_nova: 0it [00:00, ?it/s]
Other test: 100%|██████████| 29/29 [00:00<00:00, 91.89it/s]


...

========== 分类中没有类缺失！ ==========


GDP-Man test: 100%|██████████| 55/55 [00:00<00:00, 140.46it/s]
dTDP-Rha_nova: 0it [00:00, ?it/s]
Other test: 100%|██████████| 44/44 [00:00<00:00, 114.00it/s]


========== 分类中没有类缺失！ ==========


GDP-Man test: 100%|██████████| 46/46 [00:00<00:00, 117.62it/s]
dTDP-Rha_nova: 0it [00:00, ?it/s]
Other test: 100%|██████████| 34/34 [00:00<00:00, 91.73it/s] 


========== 分类中没有类缺失！ ==========


# 划分数据集（GTB)

In [84]:
import os
from Bio import SeqIO

# 生成替身序列，防止cdhit出毛病

original_fasta = './data/cluster/GTB/sequences.fasta'
index_fasta = './data/cluster/GTB/index_fasta.fasta'
index_dict = './data/cluster/GTB/index_dict.dict'

with open(original_fasta, "r") as f:
    sequences = SeqIO.parse(f, "fasta")
    sequence_dict = {seq_record.id : seq_record.seq for seq_record in sequences}

# 生成替身序列，并记录替身字典
f = open(index_fasta, 'w')
f_dict = open(index_dict, 'w')
key_list = list(sequence_dict.keys())
for i in range(0,len(key_list)):
    f_dict.write("seq{:06d}==={}\n".format(i, key_list[i]))
    f.write(">seq{:06d}\n{}\n".format(i, sequence_dict[key_list[i]]))
f.close()
f_dict.close()

print(f"cdhit -c 0.7 -i index_fasta.fasta -o index_fasta_cdhit.fasta -T 192 -M 20000")

# 在/home/admin123/work/GTmining/data/cluster/GTB/下运行


cdhit -c 0.7 -i index_fasta.fasta -o index_fasta_cdhit.fasta -T 192 -M 20000


In [6]:
# 采样获取训练集：验证集：测试集=7：2：1，单独取样nova数据集（成家族取样）
import os
import pandas as pd
import random
from tqdm import tqdm

cluster_file = './data/cluster/GTB/index_fasta_cdhit.fasta.clstr'
index_dict_file = './data/cluster/GTB/index_dict.dict'
nova_family = ['GT1', 'GT23', 'GT33'] # GTA

# 读取家族信息
df = pd.read_excel('./data/GTB_training_data.xlsx')
family_dict = {}
for i in range(0, df.shape[0]):
    family_dict[df['Family'][i]+'_'+df['GenBank'][i]] = df['Family'][i]

# 活性标签
# activate_index = {'UDP-Glc': 0, 'UDP-GlcNAc': 1, 'UDP-GlcA': 2,
#                   'UDP-Gal': 3, 'UDP-GalNAc': 4,
#                   'UDP-Xyl': 5, 'GDP-Man': 6,
#                   'dTDP-Rha': 7, 'Other': 8}
activate_index = {'UDP-Glc': 0, 'UDP-GlcNAc': 1, 'UDP-GlcA': 2,
                  'UDP-Gal': 3, 'UDP-GalNAc': 4,
                  'UDP-Xyl': 5, 'GDP-Man': 6, 'GDP-Fuc': 7,
                  'dTDP-Rha': 8, 'Other': 9}


# 获取活性标签
df = pd.read_excel('./data/GTB_training_data.xlsx')
activate_dict = {}
for i in range(0,df.shape[0]):
    if '[-]' in df['NDP-Sugar活性'][i]:
        activate_dict[df['Family'][i]+'_'+df['GenBank'][i]] = 'Bifunction'
    elif '[*]' in df['NDP-Sugar活性'][i]:
        activate_dict[df['Family'][i]+'_'+df['GenBank'][i]] = 'Other'
    else:
        activate_dict[df['Family'][i]+'_'+df['GenBank'][i]] = df['NDP-Sugar活性'][i]

# 读取cluster文件
# 读取index文件
index_dict = {}
with open(index_dict_file, 'r')as f:
    for line in f:
        line = line.strip().split('===')
        index_dict[line[0]] = line[1]
# 转译cluster文件
clusters = {}
with open(cluster_file, 'r') as file:
    current_cluster = None
    for line in file:
        line = line.strip()
        if line.startswith('>Cluster'):
            # 新的cluster开始
            cluster_number = 'Cluster_' + line.split()[1]
            current_cluster = cluster_number
            clusters[current_cluster] = []
        elif line and current_cluster is not None:
            # 这一行包含序列信息
            seq_name = line.split(', >')[1].split('... ')[0]
            clusters[current_cluster].append(index_dict[seq_name])

# 检查cluster中活性数据是否冲突
for _, cluster_list in clusters.items():
    cluster_activate = [activate_dict[x] for x in cluster_list]
    cluster_activate = list(set(cluster_activate))
    # if len(cluster_activate) >= 2:
    #     print('┗|｀O′|┛ 嗷~~，发现有的簇花心了！活性不对。')
    #     print(cluster_activate, cluster_list)

# 检查cluster中家族数据是否冲突
for _, cluster_list in clusters.items():
    cluster_activate = [family_dict[x] for x in cluster_list]
    cluster_activate = list(set(cluster_activate))
    # if len(cluster_activate) >= 2:
    #     print('┗|｀O′|┛ 嗷~~，发现有的簇花心了！家族不对。')
    #     print(cluster_activate, cluster_list)

cluster_dict = {x : [] for x in activate_index.keys()}
# 写入活性cluter字典
for _, cluster_list in clusters.items():
    if activate_dict[cluster_list[0]] == 'Bifunction':
        # 双功能的酶暂时不在考虑之中，因为在现有的体系里面划分到哪一类都不是很合适
        # 之前划分到other类里面可能会导致模型在学习的时候有点分不清楚
        continue
    elif activate_dict[cluster_list[0]] == 'UDP-GalA':
        # GalA因为数量实在是太少，所以不在考虑的范围之中
        continue
    elif len(list(set([activate_dict[x] for x in cluster_list]))) >= 2:
        # 有的序列有两个结构域，并且是双功能
        continue
    elif len(list(set([family_dict[x] for x in cluster_list]))) >= 2:
        # 有的序列有两个结构域，并且是双功能
        continue
    if activate_dict[cluster_list[0]] == 'Other':
        # Other是70%，能从2w条减少为5000条
        cluster_dict[activate_dict[cluster_list[0]]].append([cluster_list[0]])
    else:
        cluster_dict[activate_dict[cluster_list[0]]].append([x for x in cluster_list])



fold_num = 1
while fold_num <= 10:
    # ==================== 划分训练集、测试集 ====================
    df = pd.DataFrame(columns=['Family', 'GenBank', 'Cluster_GenBank_Random', 'Activate', 'Dataset'])

    for temp_key, temp_value in cluster_dict.items():
        # 写入realtest
        new_value = []
        realtest_value = []
        for cluster in temp_value:
            if cluster[0].split('_')[0] in nova_family:
                realtest_value.append(cluster)
            else:
                new_value.append(cluster)
        temp_value = new_value
        realtest_samples = realtest_value
        # 处理realtest
        for samples in tqdm(realtest_samples, desc=temp_key+'_nova'):
            sample_family = samples[0].split('_')[0]
            for sample in samples:
                df.loc[len(df)] = [sample_family, sample, samples[0], activate_dict[sample], 'nova']


        # 计算各集合的目标范围（总样本数的±5%）
        total_samples = sum(len(c) for c in temp_value)
        train_min = 0.68 * total_samples
        train_max = 0.72 * total_samples
        validation_min = 0.18 * total_samples
        validation_max = 0.22 * total_samples
        test_min = 0.08 * total_samples
        test_max = 0.12 * total_samples
        if temp_key == 'UDP-GalNAc':
            # GalNAc只有7个簇，所以要微调
            train_min = 0.90 * total_samples
            train_max = 1.00 * total_samples
            validation_min = 0.02 * total_samples
            validation_max = 0.05 * total_samples
            test_min = 0.005 * total_samples
            test_max = 0.02 * total_samples


        total_attempts = 0
        while total_attempts < 50:
            # 随机打乱簇的顺序，之后就可以直接遍历而无需采样
            shuffled = random.sample(temp_value, len(temp_value))
            
            # 初始化集合
            train, validation, test = [], [], []
            train_size, validation_size, test_size = 0, 0, 0
            
            for cluster in shuffled:
                cluster_size = len(cluster)
                candidates = []
                
                # 确定可加入的集合
                if test_size + cluster_size <= test_max:
                    candidates.append('test')
                if validation_size + cluster_size <= validation_max:
                    candidates.append('validation')
                if train_size + cluster_size <= train_max:
                    candidates.append('train')
                
                if not candidates:
                    break  # 无法分配，本次尝试失败
                
                # 优先填充未达最小值的集合
                priorities = []
                for candidate in candidates:
                    current = {'train': train_size, 'validation': validation_size, 'test': test_size}[candidate]
                    min_req = {'train': train_min, 'validation': validation_min, 'test': test_min}[candidate]
                    max_req = {'train': train_max, 'validation': validation_max, 'test': test_max}[candidate]
                    if current < min_req and (cluster_size + current) <  max_req:
                        priority = min_req - current 
                    else:
                        priority = 100000
                    priorities.append((priority, candidate))
                
                # 选择最需要填充的集合（谁差的最少优先选谁）
                priorities.sort(reverse=False, key=lambda x: x[0])
                chosen = priorities[0][1]

                # 分配簇
                if chosen == 'train':
                    train.append(cluster)
                    train_size += cluster_size
                elif chosen == 'validation':
                    validation.append(cluster)
                    validation_size += cluster_size
                elif chosen == 'test':
                    test.append(cluster)
                    test_size += cluster_size

            
            # 验证分配结果
            if (train_min <= train_size <= train_max and
                validation_min <= validation_size <= validation_max and
                test_min <= test_size <= test_max and
                (train_size + validation_size + test_size) == total_samples):
                # 展开簇中的样本
                train_samples = train
                validation_samples = validation
                test_samples = test
                break
            
            total_attempts += 1

        if total_attempts >= 50:
            print(f"error in {temp_key}")
            break


        for samples in tqdm(train_samples, desc=temp_key + ' train'):
            sample_family = samples[0].split('_')[0]
            for sample in samples:
                if activate_dict[sample] == 'Bifunction':
                    print(sample)
                df.loc[len(df)] = [sample_family, sample, samples[0], activate_dict[sample], 'train']
        for samples in tqdm(validation_samples, desc=temp_key + ' validation'):
            sample_family = samples[0].split('_')[0]
            for sample in samples:
                if activate_dict[sample] == 'Bifunction':
                    print(sample)
                df.loc[len(df)] = [sample_family, sample, samples[0], activate_dict[sample], 'validation']
        for samples in tqdm(test_samples, desc=temp_key + ' test'):
            sample_family = samples[0].split('_')[0]
            for sample in samples:
                if activate_dict[sample] == 'Bifunction':
                    print(sample)
                df.loc[len(df)] = [sample_family, sample, samples[0], activate_dict[sample], 'test']
        
        # break
        
    df = df.drop_duplicates()
    df.reset_index(drop=True, inplace=True)

    split_name = [df['Activate'][i] + '_' + df['Dataset'][i] for i in range(0, df.shape[0])]
    if not len(list(set(split_name))) == 39:
        print('========== ┗|｀O′|┛ 嗷~~，分类中有类缺失！ ==========')
        fold_num += 1
    else:
        print('========== 分类中没有类缺失！ ==========')
        df.to_excel(f'./data/cluster/GTB/dataseat_split_{fold_num}.xlsx', index=False)
        fold_num += 1

UDP-Gal test: 100%|██████████| 3/3 [00:00<00:00, 56.72it/s]
UDP-GalNAc_nova: 0it [00:00, ?it/s]
Other test: 100%|██████████| 552/552 [00:01<00:00, 385.06it/s]


========== 分类中没有类缺失！ ==========


UDP-Gal test: 100%|██████████| 2/2 [00:00<00:00, 39.77it/s]
UDP-GalNAc_nova: 0it [00:00, ?it/s]
Other test: 100%|██████████| 552/552 [00:01<00:00, 382.21it/s]


========== 分类中没有类缺失！ ==========


UDP-Gal test: 100%|██████████| 4/4 [00:00<00:00, 73.55it/s]
UDP-GalNAc_nova: 0it [00:00, ?it/s]
Other test: 100%|██████████| 552/552 [00:01<00:00, 382.68it/s]


...

========== 分类中没有类缺失！ ==========


UDP-Gal test: 100%|██████████| 3/3 [00:00<00:00, 55.44it/s]
UDP-GalNAc_nova: 0it [00:00, ?it/s]
Other test: 100%|██████████| 552/552 [00:01<00:00, 382.50it/s]


========== 分类中没有类缺失！ ==========


UDP-Gal test: 100%|██████████| 1/1 [00:00<00:00, 17.59it/s]
UDP-GalNAc_nova: 0it [00:00, ?it/s]
Other test: 100%|██████████| 552/552 [00:01<00:00, 380.49it/s]


========== 分类中没有类缺失！ ==========


# 划分数据集（GTA）—— 使用所有数据

In [ ]:
'''
使用之前处理的数据，之前的数据没有区分nova非nova
'''

# import os
# from Bio import SeqIO

# # 生成替身序列，防止cdhit出毛病

# original_fasta = './data/cluster/GTA/sequences.fasta'
# index_fasta = './data/cluster/GTA/index_fasta.fasta'
# index_dict = './data/cluster/GTA/index_dict.dict'

# with open(original_fasta, "r") as f:
#     sequences = SeqIO.parse(f, "fasta")
#     sequence_dict = {seq_record.id : seq_record.seq for seq_record in sequences}

# # 生成替身序列，并记录替身字典
# f = open(index_fasta, 'w')
# f_dict = open(index_dict, 'w')
# key_list = list(sequence_dict.keys())
# for i in range(0,len(key_list)):
#     f_dict.write("seq{:06d}==={}\n".format(i, key_list[i]))
#     f.write(">seq{:06d}\n{}\n".format(i, sequence_dict[key_list[i]]))
# f.close()
# f_dict.close()

# print(f"cdhit -c 0.7 -i index_fasta.fasta -o index_fasta_cdhit.fasta -T 192 -M 20000")

# # 在/home/admin123/work/GTmining/data/cluster/GTA/下运行


cdhit -c 0.7 -i index_fasta.fasta -o index_fasta_cdhit.fasta -T 192 -M 20000


In [2]:
# 采样获取训练集：验证集：测试集=7：2：1，单独取样nova数据集（成家族取样）
import os
import pandas as pd
import random
from tqdm import tqdm

cluster_file = './data/cluster/GTA/index_fasta_cdhit.fasta.clstr'
index_dict_file = './data/cluster/GTA/index_dict.dict'
nova_family = [] # GTA

# 读取家族信息
df = pd.read_excel('./data/GTA_training_data.xlsx')
family_dict = {}
for i in range(0, df.shape[0]):
    family_dict[df['Family'][i]+'_'+df['GenBank'][i]] = df['Family'][i]

# 活性标签
activate_index = {'UDP-Glc': 0, 'UDP-GlcNAc': 1, 'UDP-GlcA': 2,
                  'UDP-Gal': 3, 'UDP-GalNAc': 4,
                  'UDP-Xyl': 5, 'GDP-Man': 6,
                  'dTDP-Rha': 7, 'Other': 8}
# activate_index = {'UDP-Glc': 0, 'UDP-GlcNAc': 1, 'UDP-GlcA': 2,
#                   'UDP-Gal': 3, 'UDP-GalNAc': 4,
#                   'UDP-Xyl': 5, 'GDP-Man': 6, 'GDP-Fuc': 7,
#                   'dTDP-Rha': 8, 'Other': 9}


# 获取活性标签
df = pd.read_excel('./data/GTA_training_data.xlsx')
activate_dict = {}
for i in range(0,df.shape[0]):
    if '[-]' in df['NDP-Sugar活性'][i]:
        activate_dict[df['Family'][i]+'_'+df['GenBank'][i]] = 'Bifunction'
    elif '[*]' in df['NDP-Sugar活性'][i]:
        activate_dict[df['Family'][i]+'_'+df['GenBank'][i]] = 'Other'
    else:
        activate_dict[df['Family'][i]+'_'+df['GenBank'][i]] = df['NDP-Sugar活性'][i]

# 读取cluster文件
# 读取index文件
index_dict = {}
with open(index_dict_file, 'r')as f:
    for line in f:
        line = line.strip().split('===')
        index_dict[line[0]] = line[1]
# 转译cluster文件
clusters = {}
with open(cluster_file, 'r') as file:
    current_cluster = None
    for line in file:
        line = line.strip()
        if line.startswith('>Cluster'):
            # 新的cluster开始
            cluster_number = 'Cluster_' + line.split()[1]
            current_cluster = cluster_number
            clusters[current_cluster] = []
        elif line and current_cluster is not None:
            # 这一行包含序列信息
            seq_name = line.split(', >')[1].split('... ')[0]
            clusters[current_cluster].append(index_dict[seq_name])

# 检查cluster中活性数据是否冲突
for _, cluster_list in clusters.items():
    cluster_activate = [activate_dict[x] for x in cluster_list]
    cluster_activate = list(set(cluster_activate))
    # if len(cluster_activate) >= 2:
    #     print('┗|｀O′|┛ 嗷~~，发现有的簇花心了！活性不对。')
    #     print(cluster_activate, cluster_list)

# 检查cluster中家族数据是否冲突
for _, cluster_list in clusters.items():
    cluster_activate = [family_dict[x] for x in cluster_list]
    cluster_activate = list(set(cluster_activate))
    # if len(cluster_activate) >= 2:
    #     print('┗|｀O′|┛ 嗷~~，发现有的簇花心了！家族不对。')
    #     print(cluster_activate, cluster_list)

cluster_dict = {x : [] for x in activate_index.keys()}
# 写入活性cluter字典
for _, cluster_list in clusters.items():
    if activate_dict[cluster_list[0]] == 'Bifunction':
        # 双功能的酶暂时不在考虑之中，因为在现有的体系里面划分到哪一类都不是很合适
        # 之前划分到other类里面可能会导致模型在学习的时候有点分不清楚
        continue
    elif activate_dict[cluster_list[0]] == 'UDP-GalA' or activate_dict[cluster_list[0]] == 'GDP-Fuc':
        # GalA因为数量实在是太少，所以不在考虑的范围之中
        continue
    elif len(list(set([activate_dict[x] for x in cluster_list]))) >= 2:
        # 有的序列有两个结构域，并且是双功能
        continue
    elif len(list(set([family_dict[x] for x in cluster_list]))) >= 2:
        # 有的序列有两个结构域，并且是双功能
        continue
    cluster_dict[activate_dict[cluster_list[0]]].append([x for x in cluster_list])



fold_num = 1
while fold_num <= 10:
    # ==================== 划分训练集、测试集 ====================
    df = pd.DataFrame(columns=['Family', 'GenBank', 'Cluster_GenBank_Random', 'Activate', 'Dataset'])

    for temp_key, temp_value in cluster_dict.items():
        # 写入realtest
        new_value = []
        realtest_value = []
        for cluster in temp_value:
            if cluster[0].split('_')[0] in nova_family:
                realtest_value.append(cluster)
            else:
                new_value.append(cluster)
        temp_value = new_value
        realtest_samples = realtest_value
        # 处理realtest
        for samples in tqdm(realtest_samples, desc=temp_key+'_nova'):
            sample_family = samples[0].split('_')[0]
            for sample in samples:
                df.loc[len(df)] = [sample_family, sample, samples[0], activate_dict[sample], 'nova']


        # 计算各集合的目标范围（总样本数的±5%）
        total_samples = sum(len(c) for c in temp_value)
        train_min = 0.68 * total_samples
        train_max = 0.72 * total_samples
        validation_min = 0.18 * total_samples
        validation_max = 0.22 * total_samples
        test_min = 0.08 * total_samples
        test_max = 0.12 * total_samples

        total_attempts = 0
        while total_attempts < 20:
            # 随机打乱簇的顺序，之后就可以直接遍历而无需采样
            shuffled = random.sample(temp_value, len(temp_value))
            
            # 初始化集合
            train, validation, test = [], [], []
            train_size, validation_size, test_size = 0, 0, 0
            
            for cluster in shuffled:
                cluster_size = len(cluster)
                candidates = []
                
                # 确定可加入的集合
                if test_size + cluster_size <= test_max:
                    candidates.append('test')
                if validation_size + cluster_size <= validation_max:
                    candidates.append('validation')
                if train_size + cluster_size <= train_max:
                    candidates.append('train')
                
                if not candidates:
                    break  # 无法分配，本次尝试失败
                
                # 优先填充未达最小值的集合
                priorities = []
                for candidate in candidates:
                    current = {'train': train_size, 'validation': validation_size, 'test': test_size}[candidate]
                    min_req = {'train': train_min, 'validation': validation_min, 'test': test_min}[candidate]
                    priority = min_req - current if current < min_req else 100000
                    priorities.append((priority, candidate))
                
                # 选择最需要填充的集合（谁差的最少优先选谁）
                priorities.sort(reverse=False, key=lambda x: x[0])
                chosen = priorities[0][1]

                # 分配簇
                if chosen == 'train':
                    train.append(cluster)
                    train_size += cluster_size
                elif chosen == 'validation':
                    validation.append(cluster)
                    validation_size += cluster_size
                elif chosen == 'test':
                    test.append(cluster)
                    test_size += cluster_size

            
            # 验证分配结果
            if (train_min <= train_size <= train_max and
                validation_min <= validation_size <= validation_max and
                test_min <= test_size <= test_max and
                (train_size + validation_size + test_size) == total_samples):
                # 展开簇中的样本
                train_samples = train
                validation_samples = validation
                test_samples = test
                break
            
            total_attempts += 1

        if total_attempts >= 20:
            print(f"error in {temp_key}")
            break
        
        for samples in tqdm(train_samples, desc=temp_key + ' train'):
            sample_family = samples[0].split('_')[0]
            for sample in samples:
                if activate_dict[sample] == 'Bifunction':
                    print(sample)
                df.loc[len(df)] = [sample_family, sample, samples[0], activate_dict[sample], 'train']
        for samples in tqdm(validation_samples, desc=temp_key + ' validation'):
            sample_family = samples[0].split('_')[0]
            for sample in samples:
                if activate_dict[sample] == 'Bifunction':
                    print(sample)
                df.loc[len(df)] = [sample_family, sample, samples[0], activate_dict[sample], 'validation']
        for samples in tqdm(test_samples, desc=temp_key + ' test'):
            sample_family = samples[0].split('_')[0]
            for sample in samples:
                if activate_dict[sample] == 'Bifunction':
                    print(sample)
                df.loc[len(df)] = [sample_family, sample, samples[0], activate_dict[sample], 'test']
        
        # break
        
    df = df.drop_duplicates()
    df.reset_index(drop=True, inplace=True)

    split_name = [df['Activate'][i] + '_' + df['Dataset'][i] for i in range(0, df.shape[0])]
    if not len(list(set(split_name))) == 27:
        print('========== ┗|｀O′|┛ 嗷~~，分类中有类缺失！ ==========')
    else:
        print('========== 分类中没有类缺失！ ==========')
        df.to_excel(f'./data/cluster/GTA_alldata/dataseat_split_{fold_num}.xlsx', index=False)
        fold_num += 1

UDP-Glc_nova: 0it [00:00, ?it/s]
UDP-Glc test: 100%|██████████| 62/62 [00:01<00:00, 48.74it/s]
UDP-GlcNAc_nova: 0it [00:00, ?it/s]
UDP-GlcNAc test: 100%|██████████| 58/58 [00:00<00:00, 111.66it/s]
UDP-GlcA_nova: 0it [00:00, ?it/s]
UDP-GlcA test: 100%|██████████| 3/3 [00:00<00:00, 45.90it/s]
UDP-Gal_nova: 0it [00:00, ?it/s]
UDP-Gal test: 100%|██████████| 12/12 [00:00<00:00, 57.43it/s]
UDP-GalNAc_nova: 0it [00:00, ?it/s]
UDP-GalNAc test: 100%|██████████| 13/13 [00:00<00:00, 193.06it/s]
UDP-Xyl_nova: 0it [00:00, ?it/s]
UDP-Xyl test: 100%|██████████| 2/2 [00:00<00:00, 63.31it/s]
GDP-Man_nova: 0it [00:00, ?it/s]
GDP-Man test: 100%|██████████| 14/14 [00:00<00:00, 37.37it/s]
dTDP-Rha_nova: 0it [00:00, ?it/s]
dTDP-Rha test: 100%|██████████| 7/7 [00:00<00:00, 119.02it/s]
Other_nova: 0it [00:00, ?it/s]
Other test: 100%|██████████| 30/30 [00:00<00:00, 86.71it/s]


========== 分类中没有类缺失！ ==========


UDP-Glc_nova: 0it [00:00, ?it/s]
UDP-Glc test: 100%|██████████| 69/69 [00:01<00:00, 59.43it/s]
UDP-GlcNAc_nova: 0it [00:00, ?it/s]
UDP-GlcNAc test: 100%|██████████| 67/67 [00:00<00:00, 132.29it/s]
UDP-GlcA_nova: 0it [00:00, ?it/s]
UDP-GlcA test: 100%|██████████| 2/2 [00:00<00:00, 31.72it/s]
UDP-Gal_nova: 0it [00:00, ?it/s]
UDP-Gal test: 100%|██████████| 6/6 [00:00<00:00, 25.86it/s]
UDP-GalNAc_nova: 0it [00:00, ?it/s]
UDP-GalNAc test: 100%|██████████| 20/20 [00:00<00:00, 264.31it/s]
UDP-Xyl_nova: 0it [00:00, ?it/s]
UDP-Xyl test: 100%|██████████| 1/1 [00:00<00:00, 30.56it/s]
GDP-Man_nova: 0it [00:00, ?it/s]
GDP-Man test: 100%|██████████| 35/35 [00:00<00:00, 84.90it/s]
dTDP-Rha_nova: 0it [00:00, ?it/s]
dTDP-Rha test: 100%|██████████| 8/8 [00:00<00:00, 130.55it/s]
Other_nova: 0it [00:00, ?it/s]
Other test: 100%|██████████| 4/4 [00:00<00:00, 10.36it/s]


========== 分类中没有类缺失！ ==========


UDP-Glc_nova: 0it [00:00, ?it/s]
UDP-Glc test: 100%|██████████| 47/47 [00:01<00:00, 43.89it/s]
UDP-GlcNAc_nova: 0it [00:00, ?it/s]
UDP-GlcNAc test: 100%|██████████| 66/66 [00:00<00:00, 130.21it/s]
UDP-GlcA_nova: 0it [00:00, ?it/s]
UDP-GlcA test: 100%|██████████| 3/3 [00:00<00:00, 46.76it/s]
UDP-Gal_nova: 0it [00:00, ?it/s]
UDP-Gal test: 100%|██████████| 12/12 [00:00<00:00, 72.14it/s]
UDP-GalNAc_nova: 0it [00:00, ?it/s]
UDP-GalNAc test: 100%|██████████| 11/11 [00:00<00:00, 168.39it/s]
UDP-Xyl_nova: 0it [00:00, ?it/s]
UDP-Xyl test: 100%|██████████| 2/2 [00:00<00:00, 63.38it/s]
GDP-Man_nova: 0it [00:00, ?it/s]
GDP-Man test: 100%|██████████| 46/46 [00:00<00:00, 125.82it/s]
dTDP-Rha_nova: 0it [00:00, ?it/s]
dTDP-Rha test: 100%|██████████| 8/8 [00:00<00:00, 159.21it/s]
Other_nova: 0it [00:00, ?it/s]
Other test: 100%|██████████| 52/52 [00:00<00:00, 141.82it/s]


...

========== 分类中没有类缺失！ ==========


UDP-Glc_nova: 0it [00:00, ?it/s]
UDP-Glc test: 100%|██████████| 74/74 [00:01<00:00, 56.47it/s]
UDP-GlcNAc_nova: 0it [00:00, ?it/s]
UDP-GlcNAc test: 100%|██████████| 40/40 [00:00<00:00, 78.91it/s]
UDP-GlcA_nova: 0it [00:00, ?it/s]
UDP-GlcA test: 100%|██████████| 2/2 [00:00<00:00, 32.04it/s]
UDP-Gal_nova: 0it [00:00, ?it/s]
UDP-Gal test: 100%|██████████| 11/11 [00:00<00:00, 56.77it/s]
UDP-GalNAc_nova: 0it [00:00, ?it/s]
UDP-GalNAc test: 100%|██████████| 14/14 [00:00<00:00, 187.97it/s]
UDP-Xyl_nova: 0it [00:00, ?it/s]
UDP-Xyl test: 100%|██████████| 1/1 [00:00<00:00, 34.23it/s]
GDP-Man_nova: 0it [00:00, ?it/s]
GDP-Man test: 100%|██████████| 35/35 [00:00<00:00, 89.40it/s]
dTDP-Rha_nova: 0it [00:00, ?it/s]
dTDP-Rha test: 100%|██████████| 8/8 [00:00<00:00, 164.09it/s]
Other_nova: 0it [00:00, ?it/s]
Other test: 100%|██████████| 25/25 [00:00<00:00, 67.33it/s]


========== 分类中没有类缺失！ ==========


UDP-Glc_nova: 0it [00:00, ?it/s]
UDP-Glc test: 100%|██████████| 69/69 [00:01<00:00, 58.56it/s] 
UDP-GlcNAc_nova: 0it [00:00, ?it/s]
UDP-GlcNAc test: 100%|██████████| 53/53 [00:00<00:00, 105.10it/s]
UDP-GlcA_nova: 0it [00:00, ?it/s]
UDP-GlcA test: 100%|██████████| 1/1 [00:00<00:00, 20.50it/s]
UDP-Gal_nova: 0it [00:00, ?it/s]
UDP-Gal test: 100%|██████████| 9/9 [00:00<00:00, 41.16it/s]
UDP-GalNAc_nova: 0it [00:00, ?it/s]
UDP-GalNAc test: 100%|██████████| 24/24 [00:00<00:00, 353.93it/s]
UDP-Xyl_nova: 0it [00:00, ?it/s]
UDP-Xyl test: 100%|██████████| 2/2 [00:00<00:00, 63.30it/s]
GDP-Man_nova: 0it [00:00, ?it/s]
GDP-Man test: 100%|██████████| 31/31 [00:00<00:00, 75.00it/s] 
dTDP-Rha_nova: 0it [00:00, ?it/s]
dTDP-Rha test: 100%|██████████| 9/9 [00:00<00:00, 170.73it/s]
Other_nova: 0it [00:00, ?it/s]
Other test: 100%|██████████| 31/31 [00:00<00:00, 80.48it/s]


========== 分类中没有类缺失！ ==========


# 划分数据集（GTB）—— 使用所有数据

In [ ]:
'''
使用之前处理的数据，之前的数据没有区分nova非nova
'''

# import os
# from Bio import SeqIO

# # 生成替身序列，防止cdhit出毛病

# original_fasta = './data/cluster/GTB/sequences.fasta'
# index_fasta = './data/cluster/GTB/index_fasta.fasta'
# index_dict = './data/cluster/GTB/index_dict.dict'

# with open(original_fasta, "r") as f:
#     sequences = SeqIO.parse(f, "fasta")
#     sequence_dict = {seq_record.id : seq_record.seq for seq_record in sequences}

# # 生成替身序列，并记录替身字典
# f = open(index_fasta, 'w')
# f_dict = open(index_dict, 'w')
# key_list = list(sequence_dict.keys())
# for i in range(0,len(key_list)):
#     f_dict.write("seq{:06d}==={}\n".format(i, key_list[i]))
#     f.write(">seq{:06d}\n{}\n".format(i, sequence_dict[key_list[i]]))
# f.close()
# f_dict.close()

# print(f"cdhit -c 0.7 -i index_fasta.fasta -o index_fasta_cdhit.fasta -T 192 -M 20000")

# # 在/home/admin123/work/GTmining/data/cluster/GTB/下运行


cdhit -c 0.7 -i index_fasta.fasta -o index_fasta_cdhit.fasta -T 192 -M 20000


In [3]:
# 采样获取训练集：验证集：测试集=7：2：1，单独取样nova数据集（成家族取样）
import os
import pandas as pd
import random
from tqdm import tqdm

cluster_file = './data/cluster/GTB/index_fasta_cdhit.fasta.clstr'
index_dict_file = './data/cluster/GTB/index_dict.dict'
nova_family = [] # GTA

# 读取家族信息
df = pd.read_excel('./data/GTB_training_data.xlsx')
family_dict = {}
for i in range(0, df.shape[0]):
    family_dict[df['Family'][i]+'_'+df['GenBank'][i]] = df['Family'][i]

# 活性标签
# activate_index = {'UDP-Glc': 0, 'UDP-GlcNAc': 1, 'UDP-GlcA': 2,
#                   'UDP-Gal': 3, 'UDP-GalNAc': 4,
#                   'UDP-Xyl': 5, 'GDP-Man': 6,
#                   'dTDP-Rha': 7, 'Other': 8}
activate_index = {'UDP-Glc': 0, 'UDP-GlcNAc': 1, 'UDP-GlcA': 2,
                  'UDP-Gal': 3, 'UDP-GalNAc': 4,
                  'UDP-Xyl': 5, 'GDP-Man': 6, 'GDP-Fuc': 7,
                  'dTDP-Rha': 8, 'Other': 9}


# 获取活性标签
df = pd.read_excel('./data/GTB_training_data.xlsx')
activate_dict = {}
for i in range(0,df.shape[0]):
    if '[-]' in df['NDP-Sugar活性'][i]:
        activate_dict[df['Family'][i]+'_'+df['GenBank'][i]] = 'Bifunction'
    elif '[*]' in df['NDP-Sugar活性'][i]:
        activate_dict[df['Family'][i]+'_'+df['GenBank'][i]] = 'Other'
    else:
        activate_dict[df['Family'][i]+'_'+df['GenBank'][i]] = df['NDP-Sugar活性'][i]

# 读取cluster文件
# 读取index文件
index_dict = {}
with open(index_dict_file, 'r')as f:
    for line in f:
        line = line.strip().split('===')
        index_dict[line[0]] = line[1]
# 转译cluster文件
clusters = {}
with open(cluster_file, 'r') as file:
    current_cluster = None
    for line in file:
        line = line.strip()
        if line.startswith('>Cluster'):
            # 新的cluster开始
            cluster_number = 'Cluster_' + line.split()[1]
            current_cluster = cluster_number
            clusters[current_cluster] = []
        elif line and current_cluster is not None:
            # 这一行包含序列信息
            seq_name = line.split(', >')[1].split('... ')[0]
            clusters[current_cluster].append(index_dict[seq_name])

# 检查cluster中活性数据是否冲突
for _, cluster_list in clusters.items():
    cluster_activate = [activate_dict[x] for x in cluster_list]
    cluster_activate = list(set(cluster_activate))
    # if len(cluster_activate) >= 2:
    #     print('┗|｀O′|┛ 嗷~~，发现有的簇花心了！活性不对。')
    #     print(cluster_activate, cluster_list)

# 检查cluster中家族数据是否冲突
for _, cluster_list in clusters.items():
    cluster_activate = [family_dict[x] for x in cluster_list]
    cluster_activate = list(set(cluster_activate))
    # if len(cluster_activate) >= 2:
    #     print('┗|｀O′|┛ 嗷~~，发现有的簇花心了！家族不对。')
    #     print(cluster_activate, cluster_list)

cluster_dict = {x : [] for x in activate_index.keys()}
# 写入活性cluter字典
for _, cluster_list in clusters.items():
    if activate_dict[cluster_list[0]] == 'Bifunction':
        # 双功能的酶暂时不在考虑之中，因为在现有的体系里面划分到哪一类都不是很合适
        # 之前划分到other类里面可能会导致模型在学习的时候有点分不清楚
        continue
    elif activate_dict[cluster_list[0]] == 'UDP-GalA':
        # GalA因为数量实在是太少，所以不在考虑的范围之中
        continue
    elif len(list(set([activate_dict[x] for x in cluster_list]))) >= 2:
        # 有的序列有两个结构域，并且是双功能
        continue
    elif len(list(set([family_dict[x] for x in cluster_list]))) >= 2:
        # 有的序列有两个结构域，并且是双功能
        continue
    if activate_dict[cluster_list[0]] == 'Other':
        # Other是70%，能从2w条减少为5000条
        cluster_dict[activate_dict[cluster_list[0]]].append([cluster_list[0]])
    else:
        cluster_dict[activate_dict[cluster_list[0]]].append([x for x in cluster_list])



fold_num = 1
while fold_num <= 10:
    # ==================== 划分训练集、测试集 ====================
    df = pd.DataFrame(columns=['Family', 'GenBank', 'Cluster_GenBank_Random', 'Activate', 'Dataset'])

    for temp_key, temp_value in cluster_dict.items():
        # 写入realtest
        new_value = []
        realtest_value = []
        for cluster in temp_value:
            if cluster[0].split('_')[0] in nova_family:
                realtest_value.append(cluster)
            else:
                new_value.append(cluster)
        temp_value = new_value
        realtest_samples = realtest_value
        # 处理realtest
        for samples in tqdm(realtest_samples, desc=temp_key+'_nova'):
            sample_family = samples[0].split('_')[0]
            for sample in samples:
                df.loc[len(df)] = [sample_family, sample, samples[0], activate_dict[sample], 'nova']


        # 计算各集合的目标范围（总样本数的±5%）
        total_samples = sum(len(c) for c in temp_value)
        train_min = 0.68 * total_samples
        train_max = 0.72 * total_samples
        validation_min = 0.18 * total_samples
        validation_max = 0.22 * total_samples
        test_min = 0.08 * total_samples
        test_max = 0.12 * total_samples
        if temp_key == 'UDP-GalNAc':
            # GalNAc只有7个簇，所以要微调
            train_min = 0.90 * total_samples
            train_max = 1.00 * total_samples
            validation_min = 0.02 * total_samples
            validation_max = 0.05 * total_samples
            test_min = 0.005 * total_samples
            test_max = 0.02 * total_samples


        total_attempts = 0
        while total_attempts < 50:
            # 随机打乱簇的顺序，之后就可以直接遍历而无需采样
            shuffled = random.sample(temp_value, len(temp_value))
            
            # 初始化集合
            train, validation, test = [], [], []
            train_size, validation_size, test_size = 0, 0, 0
            
            for cluster in shuffled:
                cluster_size = len(cluster)
                candidates = []
                
                # 确定可加入的集合
                if test_size + cluster_size <= test_max:
                    candidates.append('test')
                if validation_size + cluster_size <= validation_max:
                    candidates.append('validation')
                if train_size + cluster_size <= train_max:
                    candidates.append('train')
                
                if not candidates:
                    break  # 无法分配，本次尝试失败
                
                # 优先填充未达最小值的集合
                priorities = []
                for candidate in candidates:
                    current = {'train': train_size, 'validation': validation_size, 'test': test_size}[candidate]
                    min_req = {'train': train_min, 'validation': validation_min, 'test': test_min}[candidate]
                    max_req = {'train': train_max, 'validation': validation_max, 'test': test_max}[candidate]
                    if current < min_req and (cluster_size + current) <  max_req:
                        priority = min_req - current 
                    else:
                        priority = 100000
                    priorities.append((priority, candidate))
                
                # 选择最需要填充的集合（谁差的最少优先选谁）
                priorities.sort(reverse=False, key=lambda x: x[0])
                chosen = priorities[0][1]

                # 分配簇
                if chosen == 'train':
                    train.append(cluster)
                    train_size += cluster_size
                elif chosen == 'validation':
                    validation.append(cluster)
                    validation_size += cluster_size
                elif chosen == 'test':
                    test.append(cluster)
                    test_size += cluster_size

            
            # 验证分配结果
            if (train_min <= train_size <= train_max and
                validation_min <= validation_size <= validation_max and
                test_min <= test_size <= test_max and
                (train_size + validation_size + test_size) == total_samples):
                # 展开簇中的样本
                train_samples = train
                validation_samples = validation
                test_samples = test
                break
            
            total_attempts += 1

        if total_attempts >= 50:
            print(f"error in {temp_key}")
            break


        for samples in tqdm(train_samples, desc=temp_key + ' train'):
            sample_family = samples[0].split('_')[0]
            for sample in samples:
                if activate_dict[sample] == 'Bifunction':
                    print(sample)
                df.loc[len(df)] = [sample_family, sample, samples[0], activate_dict[sample], 'train']
        for samples in tqdm(validation_samples, desc=temp_key + ' validation'):
            sample_family = samples[0].split('_')[0]
            for sample in samples:
                if activate_dict[sample] == 'Bifunction':
                    print(sample)
                df.loc[len(df)] = [sample_family, sample, samples[0], activate_dict[sample], 'validation']
        for samples in tqdm(test_samples, desc=temp_key + ' test'):
            sample_family = samples[0].split('_')[0]
            for sample in samples:
                if activate_dict[sample] == 'Bifunction':
                    print(sample)
                df.loc[len(df)] = [sample_family, sample, samples[0], activate_dict[sample], 'test']
        
        # break
        
    df = df.drop_duplicates()
    df.reset_index(drop=True, inplace=True)

    split_name = [df['Activate'][i] + '_' + df['Dataset'][i] for i in range(0, df.shape[0])]
    if not len(list(set(split_name))) == 30:
        print('========== ┗|｀O′|┛ 嗷~~，分类中有类缺失！ ==========')
        fold_num += 1
    else:
        print('========== 分类中没有类缺失！ ==========')
        df.to_excel(f'./data/cluster/GTB_alldata/dataseat_split_{fold_num}.xlsx', index=False)
        fold_num += 1

UDP-Glc_nova: 0it [00:00, ?it/s]
UDP-Glc test: 100%|██████████| 54/54 [00:01<00:00, 44.77it/s]
UDP-GlcNAc_nova: 0it [00:00, ?it/s]
UDP-GlcNAc test: 100%|██████████| 9/9 [00:00<00:00, 17.37it/s]
UDP-GlcA_nova: 0it [00:00, ?it/s]
UDP-GlcA test: 100%|██████████| 12/12 [00:00<00:00, 104.65it/s]
UDP-Gal_nova: 0it [00:00, ?it/s]
UDP-Gal test: 100%|██████████| 3/3 [00:00<00:00, 49.52it/s]
UDP-GalNAc_nova: 0it [00:00, ?it/s]
UDP-GalNAc test: 100%|██████████| 1/1 [00:00<00:00, 23.36it/s]
UDP-Xyl_nova: 0it [00:00, ?it/s]
UDP-Xyl test: 100%|██████████| 4/4 [00:00<00:00, 189.29it/s]
GDP-Man_nova: 0it [00:00, ?it/s]
GDP-Man test: 100%|██████████| 24/24 [00:00<00:00, 53.22it/s]
GDP-Fuc_nova: 0it [00:00, ?it/s]
GDP-Fuc test: 100%|██████████| 130/130 [00:00<00:00, 238.25it/s]
dTDP-Rha_nova: 0it [00:00, ?it/s]
dTDP-Rha test: 100%|██████████| 85/85 [00:00<00:00, 114.14it/s]
Other_nova: 0it [00:00, ?it/s]
Other test: 100%|██████████| 558/558 [00:01<00:00, 376.44it/s]


========== 分类中没有类缺失！ ==========


UDP-Glc_nova: 0it [00:00, ?it/s]
UDP-Glc test: 100%|██████████| 69/69 [00:00<00:00, 76.43it/s]
UDP-GlcNAc_nova: 0it [00:00, ?it/s]
UDP-GlcNAc test: 100%|██████████| 8/8 [00:00<00:00, 14.60it/s]
UDP-GlcA_nova: 0it [00:00, ?it/s]
UDP-GlcA test: 100%|██████████| 12/12 [00:00<00:00, 97.43it/s]
UDP-Gal_nova: 0it [00:00, ?it/s]
UDP-Gal test: 100%|██████████| 2/2 [00:00<00:00, 37.46it/s]
UDP-GalNAc_nova: 0it [00:00, ?it/s]
UDP-GalNAc test: 100%|██████████| 4/4 [00:00<00:00, 77.65it/s]
UDP-Xyl_nova: 0it [00:00, ?it/s]
UDP-Xyl test: 100%|██████████| 8/8 [00:00<00:00, 441.60it/s]
GDP-Man_nova: 0it [00:00, ?it/s]
GDP-Man test: 100%|██████████| 40/40 [00:00<00:00, 88.96it/s] 
GDP-Fuc_nova: 0it [00:00, ?it/s]
GDP-Fuc test: 100%|██████████| 146/146 [00:00<00:00, 268.97it/s]
dTDP-Rha_nova: 0it [00:00, ?it/s]
dTDP-Rha test: 100%|██████████| 49/49 [00:00<00:00, 66.13it/s]
Other_nova: 0it [00:00, ?it/s]
Other test: 100%|██████████| 558/558 [00:01<00:00, 375.26it/s]


========== 分类中没有类缺失！ ==========


UDP-Glc_nova: 0it [00:00, ?it/s]
UDP-Glc test: 100%|██████████| 48/48 [00:01<00:00, 38.88it/s]
UDP-GlcNAc_nova: 0it [00:00, ?it/s]
UDP-GlcNAc test: 100%|██████████| 11/11 [00:00<00:00, 21.26it/s]
UDP-GlcA_nova: 0it [00:00, ?it/s]
UDP-GlcA test: 100%|██████████| 11/11 [00:00<00:00, 85.82it/s]
UDP-Gal_nova: 0it [00:00, ?it/s]
UDP-Gal test: 100%|██████████| 4/4 [00:00<00:00, 67.74it/s]
UDP-GalNAc_nova: 0it [00:00, ?it/s]
UDP-GalNAc test: 100%|██████████| 3/3 [00:00<00:00, 62.10it/s]
UDP-Xyl_nova: 0it [00:00, ?it/s]
UDP-Xyl test: 100%|██████████| 8/8 [00:00<00:00, 437.42it/s]
GDP-Man_nova: 0it [00:00, ?it/s]
GDP-Man test: 100%|██████████| 29/29 [00:00<00:00, 92.05it/s]
GDP-Fuc_nova: 0it [00:00, ?it/s]
GDP-Fuc test: 100%|██████████| 113/113 [00:00<00:00, 211.52it/s]
dTDP-Rha_nova: 0it [00:00, ?it/s]
dTDP-Rha test: 100%|██████████| 53/53 [00:00<00:00, 73.86it/s] 
Other_nova: 0it [00:00, ?it/s]
Other test: 100%|██████████| 558/558 [00:01<00:00, 376.26it/s]


...

========== 分类中没有类缺失！ ==========


UDP-Glc_nova: 0it [00:00, ?it/s]
UDP-Glc test: 100%|██████████| 32/32 [00:01<00:00, 29.44it/s]
UDP-GlcNAc_nova: 0it [00:00, ?it/s]
UDP-GlcNAc test: 100%|██████████| 16/16 [00:00<00:00, 27.48it/s]
UDP-GlcA_nova: 0it [00:00, ?it/s]
UDP-GlcA test: 100%|██████████| 9/9 [00:00<00:00, 70.30it/s]
UDP-Gal_nova: 0it [00:00, ?it/s]
UDP-Gal test: 100%|██████████| 2/2 [00:00<00:00, 32.70it/s]
UDP-GalNAc_nova: 0it [00:00, ?it/s]
UDP-GalNAc test: 100%|██████████| 4/4 [00:00<00:00, 84.81it/s]
UDP-Xyl_nova: 0it [00:00, ?it/s]
UDP-Xyl test: 100%|██████████| 7/7 [00:00<00:00, 330.79it/s]
GDP-Man_nova: 0it [00:00, ?it/s]
GDP-Man test: 100%|██████████| 31/31 [00:00<00:00, 68.31it/s] 
GDP-Fuc_nova: 0it [00:00, ?it/s]
GDP-Fuc test: 100%|██████████| 139/139 [00:00<00:00, 251.13it/s]
dTDP-Rha_nova: 0it [00:00, ?it/s]
dTDP-Rha test: 100%|██████████| 69/69 [00:00<00:00, 101.06it/s]
Other_nova: 0it [00:00, ?it/s]
Other test: 100%|██████████| 558/558 [00:01<00:00, 378.95it/s]


========== 分类中没有类缺失！ ==========


UDP-Glc_nova: 0it [00:00, ?it/s]
UDP-Glc test: 100%|██████████| 39/39 [00:01<00:00, 31.29it/s]
UDP-GlcNAc_nova: 0it [00:00, ?it/s]
UDP-GlcNAc test: 100%|██████████| 10/10 [00:00<00:00, 18.35it/s]
UDP-GlcA_nova: 0it [00:00, ?it/s]
UDP-GlcA test: 100%|██████████| 13/13 [00:00<00:00, 107.13it/s]
UDP-Gal_nova: 0it [00:00, ?it/s]
UDP-Gal test: 100%|██████████| 4/4 [00:00<00:00, 56.12it/s]
UDP-GalNAc_nova: 0it [00:00, ?it/s]
UDP-GalNAc test: 100%|██████████| 4/4 [00:00<00:00, 77.59it/s]
UDP-Xyl_nova: 0it [00:00, ?it/s]
UDP-Xyl test: 100%|██████████| 6/6 [00:00<00:00, 327.48it/s]
GDP-Man_nova: 0it [00:00, ?it/s]
GDP-Man test: 100%|██████████| 19/19 [00:00<00:00, 53.00it/s]
GDP-Fuc_nova: 0it [00:00, ?it/s]
GDP-Fuc test: 100%|██████████| 142/142 [00:00<00:00, 260.05it/s]
dTDP-Rha_nova: 0it [00:00, ?it/s]
dTDP-Rha test: 100%|██████████| 76/76 [00:00<00:00, 126.44it/s]
Other_nova: 0it [00:00, ?it/s]
Other test: 100%|██████████| 558/558 [00:01<00:00, 383.30it/s]


========== 分类中没有类缺失！ ==========
